# AFRICA GIANTS — Continuous Training on Kaggle

Fine-tunes **McGill-NLP/AfriqueLlama-8B** (Llama 3.1 8B pre-trained on 20 African languages
including Swahili) on scraped and synthetic Tanzanian business/regulatory data.

Uses **Unsloth** for 2× faster QLoRA training with native Llama 3.1 support.
Pushes the LoRA adapter to HuggingFace Hub tagged as an African-language fine-tune.

In [ ]:
# Install Unsloth (includes compatible versions of transformers, peft, trl, bitsandbytes)
# Unsloth nightly has the best Llama 3.1 support
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub

In [ ]:
import os
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import HfApi, create_repo, login, whoami

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"BF16 supported: {is_bfloat16_supported()}")

In [ ]:
# Login to Hugging Face using the token stored in Kaggle Secrets.
# Kaggle secret label must be exactly: AFRICA_GIANTS
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("AFRICA_GIANTS")
login(token=hf_token)

hf_user = whoami(token=hf_token)["name"]
print(f"Logged in to Hugging Face as: {hf_user}")

In [ ]:
# ── Hardcoded repo references ──────────────────────────────────────────────
BASE_MODEL        = "McGill-NLP/AfriqueLlama-8B"   # Llama 3.1 8B, 20 African langs incl. Swahili
ADAPTER_REPO      = "prospaprospa007/africa-giants-adapter-v1"
MERGED_MODEL_REPO = "prospaprospa007/africa-giants-model-v1"
DATASET_REPO      = "prospaprospa007/africa-giants-dataset"

MAX_SEQ_LENGTH    = 2048
SMOKE_TEST        = True   # Set False for full training run
LOSS_THRESHOLD    = 2.2

# Ensure output repos exist
api = HfApi(token=hf_token)
for repo_id, repo_type in [
    (ADAPTER_REPO,      "model"),
    (MERGED_MODEL_REPO, "model"),
    (DATASET_REPO,      "dataset"),
]:
    create_repo(repo_id=repo_id, repo_type=repo_type, private=True, exist_ok=True, token=hf_token)
    print(f"Ready: {repo_type} repo {repo_id}")

In [ ]:
# ── Load base model via Unsloth (4-bit QLoRA, Llama 3.1 native) ───────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto: bfloat16 on Ampere+, float16 otherwise
    load_in_4bit=True,   # QLoRA
    token=hf_token,
)

# Apply Llama 3.1 chat template so the tokenizer formats prompts correctly
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

print(f"Loaded: {BASE_MODEL}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# ── Attach LoRA adapters via Unsloth ──────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM than standard
    random_state=3407,
    use_rslora=False,
)
model.print_trainable_parameters()

In [ ]:
# ── Load dataset and format with Llama 3.1 chat template ─────────────────
print(f"Loading dataset: {DATASET_REPO}")
raw_dataset = load_dataset(DATASET_REPO, token=hf_token)
print(raw_dataset)

SYSTEM_PROMPT = (
    "Wewe ni msaidizi wa AI wa biashara za Tanzania. "
    "Unajibu maswali kuhusu sheria za biashara, kodi, usajili wa kampuni, "
    "na kanuni za kifedha kwa Kiswahili na Kiingereza. "
    "You are a Tanzanian business AI assistant. Answer questions about "
    "business regulations, tax, company registration, and financial rules "
    "in both Swahili and English."
)

def format_example(example):
    """Convert instruction/input/output fields to Llama 3.1 chat format."""
    inst = example.get("instruction", "")
    ctx  = example.get("input", "") or ""
    out  = example.get("output", "")
    user_msg = f"Context: {ctx}\n\n{inst}" if ctx.strip() else inst
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": out},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_dataset = raw_dataset["train"].map(format_example, batched=False)
eval_dataset  = raw_dataset.get("validation", raw_dataset["train"].select(range(min(50, len(raw_dataset["train"])))))
if hasattr(eval_dataset, "map"):
    eval_dataset = eval_dataset.map(format_example, batched=False)

print(f"Train examples: {len(train_dataset)}")
print("Sample:\n", train_dataset[0]["text"][:400])

In [ ]:
# ── SFTTrainer with Unsloth-optimised settings ────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="./outputs",
        per_device_train_batch_size=1 if SMOKE_TEST else 2,
        gradient_accumulation_steps=1 if SMOKE_TEST else 4,
        warmup_steps=5,
        max_steps=10 if SMOKE_TEST else -1,
        num_train_epochs=1 if SMOKE_TEST else 3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none",
        save_strategy="epoch",
        evaluation_strategy="steps",
        eval_steps=50 if not SMOKE_TEST else 5,
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"Training done. Runtime: {trainer_stats.metrics['train_runtime']:.1f}s")

In [ ]:
# ── Validation loss gate ──────────────────────────────────────────────────
eval_results = trainer.evaluate()
validation_loss = eval_results.get("eval_loss", 999.0)
print(f"Validation Loss: {validation_loss:.4f} (threshold: {LOSS_THRESHOLD})")

gate_passed = validation_loss <= LOSS_THRESHOLD
print(f"Gate {'PASSED' if gate_passed else 'FAILED'}")

In [ ]:
# ── Push LoRA adapter to HF Hub with African-language model tags ──────────
if gate_passed:
    print(f"Pushing LoRA adapter to {ADAPTER_REPO}...")

    # Unsloth push: saves adapter weights only (not 8B base)
    model.push_to_hub_merged(
        ADAPTER_REPO,
        tokenizer,
        save_method="lora",
        token=hf_token,
    )

    # Update model card with correct African-language fine-tune tags
    model_card_content = f"""---
language:
- sw
- en
license: llama3.1
base_model: {BASE_MODEL}
tags:
- llama-3.1
- african-languages
- swahili
- tanzanian-business
- qlora
- unsloth
- peft
- lora
pipeline_tag: text-generation
---

# Africa Giants — Tanzanian Business AI (LoRA Adapter)

QLoRA fine-tune of [McGill-NLP/AfriqueLlama-8B](https://huggingface.co/McGill-NLP/AfriqueLlama-8B)
on Tanzanian business, tax, company registration, and financial regulation data.

**Base model:** Llama 3.1 8B pre-trained on 20 African languages including Swahili.  
**Languages:** Swahili (sw), English (en)  
**Training:** QLoRA r=16 via Unsloth on Kaggle GPU  
**Validation loss:** {validation_loss:.4f}

## Usage
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="{ADAPTER_REPO}",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```
"""
    api.upload_file(
        path_or_fileobj=model_card_content.encode(),
        path_in_repo="README.md",
        repo_id=ADAPTER_REPO,
        repo_type="model",
        token=hf_token,
    )
    print(f"Adapter and model card pushed to {ADAPTER_REPO}")
else:
    print(f"Validation loss {validation_loss:.4f} > threshold {LOSS_THRESHOLD}. Skipping push.")

In [ ]:
# ── Optional: merge adapter into base and push full model ─────────────────
# Set True only after the adapter push works cleanly.
MERGE_AND_PUSH = False

if MERGE_AND_PUSH and gate_passed:
    print(f"Merging adapter into base model and pushing to {MERGED_MODEL_REPO}...")
    model.push_to_hub_merged(
        MERGED_MODEL_REPO,
        tokenizer,
        save_method="merged_16bit",
        token=hf_token,
    )
    print(f"Merged model pushed to {MERGED_MODEL_REPO}")
else:
    print("Skipping merged model push.")